## Importar los datos

In [19]:
import polars as pl
import pandas as pd

ruta_archivo = 'data/raw/tomtom_move_reports/segments_1_to_16_aug_2024.parquet'

df_lazy_original = pl.scan_parquet(ruta_archivo)
head = pd.DataFrame(df_lazy_original.head(48).collect())
head.to_csv('original.csv', index=False)

In [55]:
columnas_finales = [
    "station_name", 
    "date", 
    "segment_id", 
    "street_name", 
    "hour_start", 
    "hour_end", 
    "travel_time_ratio",
    "sample_size",
    "frc",
    "harmonic_avg_speed",  # Añade esto
    "distance_m",          # Añade esto
    "travel_time_std"      # Añade esto
]


In [60]:
df_plan_conteo = (
    df_plan_reducido

    .filter(
        (pl.col("station_name") == "CENTRO") 
    )
    .select([
        # Contar cuántos segment_id distintos existen
        pl.col("segment_id").n_unique().alias("total_segmentos_distintos"),
        
        # Contar cuántas street_name (calles) distintas existen
        pl.col("street_name").n_unique().alias("total_calles_distintas"),

        # Contar el número total de filas (reportes)
        pl.len().alias("total_reportes")
    ])
)

# 3. Ejecutar el plan y obtener el resultado final
# El resultado será un DataFrame de Polars con una sola fila y tres columnas.
df_conteo = df_plan_conteo.collect()

df_conteo

total_segmentos_distintos,total_calles_distintas,total_reportes
u32,u32,u32
12358,668,4745472


In [59]:

MIN_SAMPLE_SIZE = 10
MAX_FRC_PRINCIPAL = 4

# 1. Cargar el archivo en modo Lazy
df_lazy = pl.scan_parquet(ruta_archivo)


df_plan_agregado = (
    df_lazy

    .filter(
        (pl.col("station_name") == "CENTRO") 
        & (pl.col("frc") <= MAX_FRC_PRINCIPAL) 
        & (pl.col("sample_size") >= MIN_SAMPLE_SIZE)
    )
    .select(columnas_finales)
    # Paso B: Agrupar por Calle y Hora de Inicio
    .group_by(["station_name", "street_name", "hour_start", "date"])
    
    # Paso C: Calcular métricas de Congestión y Confiabilidad
    .agg([

        # Ejemplo de ponderación por Polars (requiere las columnas 'harmonic_avg_speed' y 'distance_m')
        # Suma ponderada de la velocidad armónica: SUM(harmonic_avg_speed * distance_m) / SUM(distance_m)
        (
            pl.col("harmonic_avg_speed").mul(pl.col("distance_m")).sum()
            / pl.col("distance_m").sum()
        ).alias("weighted_harmonic_speed"),  # Congestión Promedio de la Calle (TTI)
        
        pl.col("travel_time_ratio").mean().alias("avg_tti_street"),
        
        # Suma total de las muestras (Tráfico Total Estimado)
        pl.col("sample_size").sum().alias("total_sample_size"),

        # Confiabilidad Promedio (Desviación del Tiempo de Viaje)
        pl.col("travel_time_std").mean().alias("avg_travel_time_std")

        
    ])
    
    # Paso D: Ordenar por fecha y calle para mejor visualización
    .sort(["date", "station_name", "street_name","hour_start"])

)



df_head_agregado = pd.DataFrame(df_plan_agregado.collect())


,0,1,2,3,4,5,6
0,None,0,2024-08-01,52.54683,1.0,2755,1.070877
1,None,1,2024-08-01,56.47811,0.990444,1559,0.778667
2,None,2,2024-08-01,59.017929,0.978974,877,0.50641
3,None,3,2024-08-01,61.455327,0.991053,435,0.53
4,None,4,2024-08-01,59.164423,0.971923,913,0.716538


In [57]:
df_plan_agregado.head().collect()

street_name,hour_start,date,weighted_harmonic_speed,avg_tti_street,total_sample_size,avg_travel_time_std
str,i64,str,f64,f64,i64,f64
null,0,"""2024-08-01""",52.54683,1.0,2755,1.070877
null,1,"""2024-08-01""",56.47811,0.990444,1559,0.778667
null,2,"""2024-08-01""",59.017929,0.978974,877,0.50641
null,3,"""2024-08-01""",61.455327,0.991053,435,0.53
null,4,"""2024-08-01""",59.164423,0.971923,913,0.716538


columnas: ["station_name", "date", "segment_id", "street_name", "hour_start", "hour_end", "travel_time_ratio"]

### Variables disponibles

["station_name", "date", "segment_id", "new_segment_id", "frc", "street_name", "distance_m", "time_set_id", "hour_start", "hour_end", "harmonic_avg_speed", "median_speed", "avg_speed", "std_speed", "travel_time_std", "sample_size", "normalized_sample_size", "avg_travel_time", "median_travel_time", "travel_time_ratio", "speed_percentiles", "geometry_wkt"]
